This notebook trains **6 models** and combines their predictions for the lowest possible MSE:
- **2 architectures**: EfficientNet-B0 + MobileNetV3-Small
- **3 folds each**: K-Fold cross validation = 2 × 3 = 6 trained models
- **Test-Time Augmentation (TTA)**: each test image is predicted 4 times with slight variations, then averaged
- **Final prediction** = average of 6 models × 4 TTA passes = 24 predictions per image



 **Mount Google Drive and extract data**

In [ ]:
from google.colab import drive
import zipfile, os

drive.mount('/content/drive')

# Extract the competition zip from Drive into Colab's local storage
print('Extracting data...')
with zipfile.ZipFile('/content/drive/MyDrive/picar/machine-learning-in-science-ii-2026 (1).zip', 'r') as z:
    z.extractall('/content/')

print('Files found:')
print(os.listdir('/content/'))

Mounted at /content/drive
Extracting data...
Files found:
['.config', 'drive', 'train.csv', 'sample_submission.csv', 'training_data', 'test_data', 'sample_data']


In [ ]:
import pandas as pd
df = pd.read_csv('/content/train.csv')
print(df['angle'].describe())
print(df['speed'].describe())

count    14419.000000
mean         0.539106
std          0.220331
min          0.000000
25%          0.437500
50%          0.500000
75%          0.687500
max          1.000000
Name: angle, dtype: float64
count    14419.000000
mean         0.795339
std          0.403467
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: speed, dtype: float64


**Imports**

In [ ]:
import os, random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import KFold


**Configuration**

Key differences from the baseline:
- `LR = 3e-4` (lower than baseline's 1e-3 — more careful training across multiple models)
- `EPOCHS = 50` (more epochs per fold)
- `PATIENCE = 10` (more tolerant early stopping)
- `N_FOLDS = 3` (3-fold cross validation — trains 3 versions of each architecture)
- `TTA_STEPS = 4` (4 augmented predictions per test image at inference time)
- `ANGLE_W / SPEED_W` (separate loss weights — increase ANGLE_W if angle predictions are worse)

In [ ]:
SEED         = 42
IMG_H        = 120
IMG_W        = 160
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 10
N_FOLDS      = 3
TTA_STEPS    = 4
ANGLE_W      = 1.0     # loss weight for angle — increase if angle MSE is high
SPEED_W      = 1.0     # loss weight for speed — increase if speed MSE is high

TRAIN_CSV  = '/content/train.csv'
TRAIN_DIR  = '/content/training_data/training_data'
TEST_DIR   = '/content/test_data/test_data'
OUTPUT_CSV = '/content/submission_ensemble.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'Using device: {DEVICE}')
print(f'Total models to train: {N_FOLDS * 2} ({N_FOLDS} folds x 2 architectures)')

Using device: cuda
Total models to train: 6 (3 folds x 2 architectures)


**Dataset class**

Same as baseline with corrupted image filtering built in.
Any image that can't be opened is automatically skipped.

In [ ]:
class CarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.img_dir   = Path(img_dir)
        self.transform = transform
        self.is_test   = is_test

        # Filter out corrupted images upfront
        valid_rows = []
        for _, row in df.iterrows():
            img_path = self.img_dir / f"{int(row['image_id'])}.png"
            try:
                with Image.open(img_path) as img:
                    img.verify()
                valid_rows.append(row)
            except Exception:
                pass  # silently skip corrupted images

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)
        print(f'  Dataset ready: {len(self.df)} valid images')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self.img_dir / f"{int(row['image_id'])}.png"
        img      = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img, int(row['image_id'])
        label = torch.tensor([float(row['angle']), float(row['speed'])], dtype=torch.float32)
        return img, label

**Transforms**

Three transform sets:
- `train_tf()` — aggressive augmentation during training
- `val_tf()` — no augmentation for clean validation measurement
- `tta_tf(i)` — 4 slightly different augmentations for test-time averaging

In [ ]:
NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def train_tf():
    """Augmented transforms for training — teaches robustness to lighting and position."""
    return transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.06),
        transforms.RandomAffine(degrees=4, translate=(0.04, 0.04), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize(**NORM),
    ])

def val_tf():
    """Clean transforms for validation — no augmentation so MSE is comparable."""
    return transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        transforms.ToTensor(),
        transforms.Normalize(**NORM),
    ])

def tta_tf(i):
    """
    Test-time augmentation transforms.
    i=0 = no jitter (clean pass)
    i=1 = slight jitter
    i=2 = moderate jitter
    i=3 = stronger jitter
    Averaging these 4 predictions reduces variance in final output.
    """
    jitter = transforms.ColorJitter(brightness=0.1 * i, contrast=0.1 * i)
    return transforms.Compose([
        transforms.Resize((IMG_H, IMG_W)),
        jitter,
        transforms.ToTensor(),
        transforms.Normalize(**NORM),
    ])

**Two model architectures**



In [ ]:
def build_efficientnet():
    """EfficientNet-B0 with custom regression head."""
    backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_f = backbone.classifier[1].in_features  # 1280 features from backbone
    backbone.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_f, 128),
        nn.SiLU(),           # smoother non-linearity than ReLU
        nn.Dropout(p=0.2),
        nn.Linear(128, 2),
        nn.Sigmoid()         # outputs in [0, 1]
    )
    return backbone


def build_mobilenet():
    """MobileNetV3-Small with custom regression head."""
    backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_f = backbone.classifier[0].in_features  # 576 features from backbone
    backbone.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_f, 128),
        nn.Hardswish(),      # efficient activation used in MobileNetV3
        nn.Dropout(p=0.2),
        nn.Linear(128, 2),
        nn.Sigmoid()
    )
    return backbone

**Loss function**


In [ ]:
def weighted_mse(pred, target):
    """MSE loss with separate weights for angle and speed."""
    w = torch.tensor([ANGLE_W, SPEED_W], device=pred.device)
    return ((pred - target) ** 2 * w).mean()


def run_epoch(model, loader, optimiser=None, scaler=None):
    """Run one epoch of training or validation. Pass optimiser=None for validation."""
    is_train = optimiser is not None
    model.train() if is_train else model.eval()
    total = 0.0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, desc='  train' if is_train else '  valid', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimiser.zero_grad()
                with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                    loss = weighted_mse(model(imgs), labels)
                scaler.scale(loss).backward()
                # Gradient clipping — prevents exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimiser)
                scaler.update()
            else:
                loss = weighted_mse(model(imgs), labels)
            total += loss.item() * imgs.size(0)

    return total / len(loader.dataset)

**Single fold training function**

This function trains one model on one fold's data.
It will be called 6 times total (3 folds × 2 architectures) as mentioned above.

In [ ]:
def train_one_fold(build_fn, train_df, val_df, save_path):
    """
    Train one model on one fold.

    Parameters

    build_fn   : function that builds the model (build_efficientnet or build_mobilenet)
    train_df   : dataframe of training samples for this fold
    val_df     : dataframe of validation samples for this fold
    save_path  : where to save the best model weights

    Returns

    model      : the trained model with best weights loaded
    best_mse   : the best validation MSE achieved
    """
    print(f'  Building datasets...')
    train_ds = CarDataset(train_df, TRAIN_DIR, train_tf())
    val_ds   = CarDataset(val_df,   TRAIN_DIR, val_tf())

    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model     = build_fn().to(DEVICE)
    optimiser = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    best_mse, patience_count = float('inf'), 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_epoch(model, train_loader, optimiser, scaler)
        val_loss   = run_epoch(model, val_loader)
        scheduler.step()

        print(f'  [{epoch:3d}/{EPOCHS}] train={train_loss:.5f}  val={val_loss:.5f}  lr={scheduler.get_last_lr()[0]:.1e}')

        if val_loss < best_mse:
            best_mse = val_loss
            patience_count = 0
            torch.save(model.state_dict(), save_path)
            print(f'    New best: {best_mse:.5f}')
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f'    Early stopping.')
                break

    # Load the best weights before returning
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return model, best_mse

**K-Fold cross validation training**


In [ ]:
# Load full training data
df = pd.read_csv(TRAIN_CSV)
print(f'Loaded {len(df)} training samples')

# Storage for all trained models and their scores
all_models   = []
all_val_mses = []

# KFold splits the data into N_FOLDS chunks
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Train each architecture across all folds
for arch_name, build_fn in [('efficientnet', build_efficientnet),
                              ('mobilenet',    build_mobilenet)]:

    print(f'Architecture: {arch_name.upper()}')

    for fold, (train_idx, val_idx) in enumerate(kf.split(df), 1):
        print(f'\n   Fold {fold}/{N_FOLDS} ')
        print(f'  Train samples: {len(train_idx)}  Val samples: {len(val_idx)}')

        save_path = f'/content/drive/MyDrive/picar/{arch_name}_fold{fold}.pt'

        model, val_mse = train_one_fold(
            build_fn,
            df.iloc[train_idx],
            df.iloc[val_idx],
            save_path
        )

        all_models.append(model)
        all_val_mses.append(val_mse)
        print(f'  Fold {fold} best val MSE: {val_mse:.6f}')

print(f'All {len(all_models)} models trained!')
print(f'Individual val MSEs: {[f"{m:.5f}" for m in all_val_mses]}')
print(f'Mean val MSE: {np.mean(all_val_mses):.6f}')

Loaded 14419 training samples
Architecture: EFFICIENTNET

   Fold 1/3 
  Train samples: 9612  Val samples: 4807
  Building datasets...
  Dataset ready: 9588 valid images
  Dataset ready: 4795 valid images
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 


  [  1/50] train=0.04696  val=0.01846  lr=3.0e-04
    New best: 0.01846


  [  2/50] train=0.02153  val=0.01680  lr=3.0e-04
    New best: 0.01680


  [  3/50] train=0.01765  val=0.01507  lr=3.0e-04
    New best: 0.01507


...

  [ 17/50] train=0.00892  val=0.01271  lr=2.2e-04


  [ 18/50] train=0.00907  val=0.01260  lr=2.1e-04


  [ 19/50] train=0.00854  val=0.01293  lr=2.1e-04


  [ 20/50] train=0.00846  val=0.01256  lr=2.0e-04


  [ 21/50] train=0.00855  val=0.01243  lr=1.9e-04
    Early stopping.
  Fold 3 best val MSE: 0.012253
All 6 models trained!
Individual val MSEs: ['0.01113', '0.01140', '0.01093', '0.01105', '0.01208', '0.01225']
Mean val MSE: 0.011475


**Test-Time Augmentation (TTA) + Ensemble inference**

In [ ]:
# Collect all test image IDs
test_ids = sorted([int(f.stem) for f in Path(TEST_DIR).glob('*.png')])
test_df  = pd.DataFrame({'image_id': test_ids})
print(f'Test images found: {len(test_ids)}')

# Accumulator for predictions across all TTA passes
all_preds = np.zeros((len(test_ids), 2), dtype=np.float32)

for tta_i in range(TTA_STEPS):
    print(f'\nTTA pass {tta_i + 1}/{TTA_STEPS}...')

    tf  = tta_tf(tta_i)
    ds  = CarDataset(test_df, TEST_DIR, transform=tf, is_test=True)
    ldr = DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=2)

    tta_preds = np.zeros((len(test_ids), 2), dtype=np.float32)
    ptr = 0

    for imgs, _ in tqdm(ldr, desc=f'  Predicting (TTA {tta_i+1})'):
        imgs = imgs.to(DEVICE)
        batch_pred = np.zeros((imgs.size(0), 2), dtype=np.float32)

        # Average predictions from ALL 6 models for this batch
        for m in all_models:
            m.eval()
            with torch.no_grad():
                batch_pred += m(imgs).cpu().numpy()
        batch_pred /= len(all_models)  # average across models

        tta_preds[ptr:ptr + imgs.size(0)] = batch_pred
        ptr += imgs.size(0)

    all_preds += tta_preds  # accumulate across TTA passes

# Average across all TTA passes
all_preds /= TTA_STEPS
all_preds  = np.clip(all_preds, 0, 1)  # safety clip to [0,1]

print(f'\nEnsemble predictions complete!')
print(f'Models used: {len(all_models)}')
print(f'TTA passes:  {TTA_STEPS}')
print(f'Total predictions averaged per image: {len(all_models) * TTA_STEPS}')

Test images found: 2000

TTA pass 1/4...
  Dataset ready: 2000 valid images


  Predicting (TTA 1): 100%|██████████| 63/63 [00:10<00:00,  6.10it/s]



TTA pass 2/4...
  Dataset ready: 2000 valid images


  Predicting (TTA 2): 100%|██████████| 63/63 [00:11<00:00,  5.72it/s]



TTA pass 3/4...
  Dataset ready: 2000 valid images


  Predicting (TTA 3): 100%|██████████| 63/63 [00:09<00:00,  6.73it/s]



TTA pass 4/4...
  Dataset ready: 2000 valid images


  Predicting (TTA 4): 100%|██████████| 63/63 [00:10<00:00,  5.99it/s]


Ensemble predictions complete!
Models used: 6
TTA passes:  4
Total predictions averaged per image: 24


In [ ]:

TRAIN_CSV = '/content/train.csv'
TRAIN_DIR = '/content/training_data/training_data'
IMG_H, IMG_W = 120, 160
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

WEIGHTS = {
    'efficientnet_fold1': '/content/drive/MyDrive/picar/efficientnet_fold1.pt',
    'efficientnet_fold2': '/content/drive/MyDrive/picar/efficientnet_fold2.pt',
    'efficientnet_fold3': '/content/drive/MyDrive/picar/efficientnet_fold3.pt',
    'mobilenet_fold1':    '/content/drive/MyDrive/picar/mobilenet_fold1.pt',
    'mobilenet_fold2':    '/content/drive/MyDrive/picar/mobilenet_fold2.pt',
    'mobilenet_fold3':    '/content/drive/MyDrive/picar/mobilenet_fold3.pt',
}

# Model definitions
def build_efficientnet():
    backbone = models.efficientnet_b0(weights=None)
    in_f = backbone.classifier[1].in_features
    backbone.classifier = nn.Sequential(
        nn.Dropout(p=0.3), nn.Linear(in_f, 128),
        nn.SiLU(), nn.Dropout(p=0.2),
        nn.Linear(128, 2), nn.Sigmoid()
    )
    return backbone

def build_mobilenet():
    backbone = models.mobilenet_v3_small(weights=None)
    in_f = backbone.classifier[0].in_features
    backbone.classifier = nn.Sequential(
        nn.Dropout(p=0.3), nn.Linear(in_f, 128),
        nn.Hardswish(), nn.Dropout(p=0.2),
        nn.Linear(128, 2), nn.Sigmoid()
    )
    return backbone

# Dataset
class CarDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.img_dir = Path(img_dir)
        self.transform = transform
        valid_rows = []
        for _, row in df.iterrows():
            img_path = self.img_dir / f"{int(row['image_id'])}.png"
            try:
                with Image.open(img_path) as img:
                    img.verify()
                valid_rows.append(row)
            except Exception:
                pass
        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / f"{int(row['image_id'])}.png").convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor([float(row['angle']), float(row['speed'])], dtype=torch.float32)
        return img, label

# Evaluation
val_tf = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(**NORM),
])

df = pd.read_csv(TRAIN_CSV)
# Use last 20% of data as evaluation set
eval_df = df.iloc[int(len(df) * 0.8):].reset_index(drop=True)
print(f'Evaluating on {len(eval_df)} samples...\n')

dataset = CarDataset(eval_df, TRAIN_DIR, val_tf)
loader  = DataLoader(dataset, BATCH_SIZE, shuffle=False, num_workers=2)

results = {}

for name, path in WEIGHTS.items():
    arch = 'efficientnet' if 'efficientnet' in name else 'mobilenet'
    build_fn = build_efficientnet if arch == 'efficientnet' else build_mobilenet

    model = build_fn().to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()

    total_mse, total_angle_mse, total_speed_mse = 0.0, 0.0, 0.0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs)
            total_mse       += ((preds - labels) ** 2).mean().item() * imgs.size(0)
            total_angle_mse += ((preds[:, 0] - labels[:, 0]) ** 2).mean().item() * imgs.size(0)
            total_speed_mse += ((preds[:, 1] - labels[:, 1]) ** 2).mean().item() * imgs.size(0)

    n = len(dataset)
    results[name] = {
        'mse':       total_mse / n,
        'angle_mse': total_angle_mse / n,
        'speed_mse': total_speed_mse / n,
    }
    print(f"{name:25s} | MSE: {results[name]['mse']:.6f} | "
          f"Angle MSE: {results[name]['angle_mse']:.6f} | "
          f"Speed MSE: {results[name]['speed_mse']:.6f}")

# Pick the best
best_name = min(results, key=lambda k: results[k]['mse'])
print(f"BEST MODEL: {best_name}")
print(f"MSE:        {results[best_name]['mse']:.6f}")
print(f"Angle MSE:  {results[best_name]['angle_mse']:.6f}")
print(f"Speed MSE:  {results[best_name]['speed_mse']:.6f}")
print(f"\nBest {WEIGHTS[best_name]}")

KeyboardInterrupt: 

**Save submission and download**

In [ ]:
sub = pd.DataFrame({
    'image_id': test_ids,
    'angle':    all_preds[:, 0],
    'speed':    all_preds[:, 1],
})

sub.to_csv(OUTPUT_CSV, index=False)
print(f'Submission saved: {OUTPUT_CSV}')
print(f'Rows: {len(sub)}')
print(sub.head(10))
print('\nPrediction stats:')
print(sub[['angle', 'speed']].describe())

NameError: name 'test_ids' is not defined

In [ ]:
# Download
from google.colab import files
files.download(OUTPUT_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>